# GSE160306 human bulk-retina validation

Independent human bulk-retina validation of locked Müller signatures across disease stage and retinal region. Failed exploratory cells and execution outputs were removed.

**Important:** These notebooks document the finalized analysis logic. Public raw data are downloaded from the cited repositories; large intermediate objects and outputs are intentionally not embedded in the GitHub version. Where a dataset lacks condition-level biological replication, results are explicitly treated as descriptive.

# GSE160306 — Independent Human Bulk-Retina Validation  
## LOCKED / CLOSED ANALYSIS

**Project:** Müller adaptive-to-reactive remodeling across diabetic retinopathy progression

**Role of this dataset:** Independent human bulk-retina validation of signatures defined and frozen in the early-diabetes mouse discovery dataset GSE328966. This dataset was not used to redefine, optimize, or select the discovery signatures.

### Cohort and design

GSE160306 contains 79 retinal tissue samples from 43 donors across four donor-consistent clinical groups:

- Control
- Diabetes without clinically apparent diabetic retinopathy
- NPDR
- Advanced NPDR/PDR with DME

Macular and peripheral retina were modeled separately within a donor-aware framework. Repeated samples from the same donor were accounted for using limma/voom with duplicateCorrelation. Age, sex, RIN, and post-mortem interval were included as covariates.

The primary analysis used all 79 raw-count samples. A sensitivity analysis reproduced the analysis using the 76-sample subset represented in the authors' processed CPM matrix.

### Locked signatures tested

All signatures originated from GSE328966 and were frozen before analysis of this human dataset.

**Early transient Müller program:** 184 mouse genes.  
Of these, 168 unique human orthologs were represented in the annotation and 166 were sufficiently measurable for the bulk-retina signature test. CCL21 and PIGY were not meaningfully measurable and were excluded only for assay-coverage reasons.

**IFN-α/β module:** 6 genes.

**Retinoid-metabolism module:** 5 genes.

**ECM/GAG exploratory module:** 13 genes.

Low-expression but measurable locked genes were retained rather than removed by the generic transcriptome-wide expression filter.

### Locked result

GSE160306 does **not** provide evidence for a universally conserved human diabetes-without-retinopathy peak of the early mouse Müller adaptive program.

The broad Early184-derived human signature showed only a weak peripheral trend in diabetes without retinopathy, disappeared in NPDR, and showed a second weak increase in advanced disease. No comparable trajectory was evident in macula.

The IFN module did not reproduce an early diabetic peak and instead showed a directional late peripheral rebound in advanced disease.

The retinoid-metabolism module was generally reduced rather than increased in human diabetic retina, with the strongest nominal suppression observed in advanced macula.

The ECM/GAG module showed little evidence of early activation but displayed directional activation in advanced peripheral retina, consistent with later tissue remodeling.

No locked signature contrast remained significant after correction across the complete 40-test family.

The 76-sample sensitivity analysis produced the same qualitative conclusions and therefore did not indicate that the primary results were driven by the three additional advanced-stage samples retained in the 79-sample raw-count analysis.

### Interpretation

This human bulk-retina dataset supports **stage- and retinal-region-dependent remodeling rather than universal conservation of the early mouse adaptive Müller state**.

The absence of formal human replication is treated as a biological result, not as justification to redefine the discovery signatures.

### Stop rule

This dataset is closed.

No additional pathways, alternative gene subsets, optimized signatures, or post hoc disease-stage definitions will be explored in GSE160306 for the primary manuscript. Any future analysis must be explicitly labeled exploratory and kept separate from the locked validation framework.

In [ ]:
!pip -q install GEOparse

import GEOparse, pandas as pd

gse = GEOparse.get_GEO("GSE160306", destdir=".", silent=True)

rows = []
for gsm in gse.gsms.values():
    x = {"GSM": gsm.name}
    for s in gsm.metadata.get("characteristics_ch1", []):
        if ": " in s:
            k, v = s.split(": ", 1)
            x[k] = v
    rows.append(x)

meta = pd.DataFrame(rows)

print("Samples:", len(meta), "| Donors:", meta["donor"].nunique())
display(pd.crosstab(meta["disease_group_dme"], meta["sample_site"], margins=True))
display(meta.groupby("donor").size().value_counts().sort_index()
        .rename("donors_with_this_many_samples"))

In [ ]:
pairs = meta.pivot_table(
    index=["donor", "disease_group_dme"],
    columns="sample_site", values="GSM",
    aggfunc="count", fill_value=0
)

pairs["total"] = pairs.sum(axis=1)
display(pairs.sort_values(["disease_group_dme", "total"]))

print("Donors with both sites:",
      ((pairs["Macula"] > 0) & (pairs["Periphery"] > 0)).sum())
print("Donors missing one site:",
      ((pairs["Macula"] == 0) | (pairs["Periphery"] == 0)).sum())

In [ ]:
# Find the eye/laterality metadata field
print([c for c in meta.columns if any(x in c.lower()
      for x in ["eye", "later", "side"])])

multi = pairs[pairs["total"] > 2].reset_index()["donor"]

cols = ["GSM", "donor", "disease_group_dme", "sample_site"]
cols += [c for c in meta.columns if any(x in c.lower()
         for x in ["eye", "later", "side"])]

display(meta[meta["donor"].isin(multi)][cols].sort_values(["donor", "sample_site"]))

In [ ]:
key = ["donor", "eye", "sample_site"]

dup = meta[meta.duplicated(key, keep=False)].sort_values(key)

show = ["GSM", "donor", "eye", "sample_site", "disease_group_dme"]
show += [c for c in meta.columns if any(x in c.lower()
         for x in ["title", "source", "region", "sample"]) and c not in show]

display(dup[show])
print("Duplicated donor-eye-site units:",
      dup[key].drop_duplicates().shape[0])

In [ ]:
a = meta[meta["GSM"].isin(["GSM4871140","GSM4871144"])].set_index("GSM").T

diff = a.loc[a.nunique(axis=1, dropna=False) > 1]

display(diff)

In [ ]:
print("Supplementary files:")
for x in gse.metadata.get("supplementary_file", []):
    print(x)

In [ ]:
url = [x for x in gse.metadata["supplementary_file"] if "_counts" in x][0]
url = url.replace("ftp://", "https://")

counts = pd.read_csv(url, sep="\t", compression="gzip")

print("Count matrix shape:", counts.shape)
print("\nFirst 10 columns:")
print(counts.columns[:10].tolist())
display(counts.head(3))

In [ ]:
counts = pd.read_csv(url, sep=",", compression="gzip")

print("Shape:", counts.shape)
print("Gene column:", counts.columns[0])
print("Sample columns:", counts.shape[1] - 1)
print("First 5 samples:", counts.columns[1:6].tolist())

In [ ]:
expr_samples = set(counts.columns[1:])
meta_samples = set(meta["sampleID"])

print("Expression samples:", len(expr_samples))
print("Metadata samples:", len(meta_samples))
print("In counts, not metadata:", sorted(expr_samples - meta_samples))
print("In metadata, not counts:", sorted(meta_samples - expr_samples))

all_nums = {f"sample_{i:02d}" for i in range(1, 81)}
print("Missing original sample number:", sorted(all_nums - expr_samples))

In [ ]:
gene = counts["ensemblID"].astype(str)
X = counts.iloc[:, 1:]

print("Genes:", len(gene))
print("Duplicate Ensembl IDs:", gene.duplicated().sum())
print("IDs with version suffix:", gene.str.contains(r"\.\d+$").sum())
print("All-zero genes:", (X.sum(axis=1) == 0).sum())
print("Negative counts:", (X < 0).sum().sum())

lib = X.sum(axis=0)
print("Library size range:", f"{lib.min():,.0f}", "to", f"{lib.max():,.0f}")

In [ ]:
cpm_url = [x for x in gse.metadata["supplementary_file"] if "normalized_cpm" in x][0]
cpm_url = cpm_url.replace("ftp://", "https://")

cpm = pd.read_csv(cpm_url, sep=",", compression="gzip")

print("CPM shape:", cpm.shape)
print("Same genes:", cpm["ensemblID"].equals(counts["ensemblID"]))
print("Same samples:", set(cpm.columns[1:]) == set(counts.columns[1:]))
print("Same sample order:", list(cpm.columns[1:]) == list(counts.columns[1:]))

In [ ]:
cpm_samples = set(cpm.columns[1:])
removed = sorted(expr_samples - cpm_samples)

print("Samples absent from supplied CPM:", removed)

display(
    meta[meta["sampleID"].isin(removed)]
    [["sampleID","GSM","donor","eye","sample_site","disease_group_dme"]]
    .sort_values("sampleID")
)

In [ ]:
affected = meta.loc[meta["sampleID"].isin(removed), "donor"].unique()

x = meta[meta["donor"].isin(affected)].copy()
x["in_author_CPM"] = x["sampleID"].isin(cpm_samples)

display(
    x[["sampleID","GSM","donor","eye","sample_site",
       "disease_group_dme","in_author_CPM"]]
    .sort_values(["donor","sample_site","eye"])
)

In [ ]:
print("Donors spanning >1 disease group:",
      (meta.groupby("donor")["disease_group_dme"].nunique() > 1).sum())

print("Donor-eye units spanning >1 disease group:",
      (meta.groupby(["donor","eye"])["disease_group_dme"].nunique() > 1).sum())

display(
    meta.groupby("disease_group_dme")["donor"].nunique()
        .rename("unique_donors")
)

In [ ]:
keys = ["age","sex","gender","rin","post","batch","death","race",
        "bmi","hypertension","cholesterol","medication"]

covars = [c for c in meta.columns if any(k in c.lower() for k in keys)]

print("Potential covariate columns:")
for c in covars:
    print(c)

In [ ]:
don = meta.sort_values("GSM").drop_duplicates("donor").copy()
don["age"] = pd.to_numeric(don["age"], errors="coerce")

display(
    don.groupby("disease_group_dme")
       .agg(n=("donor","size"),
            age_mean=("age","mean"),
            age_sd=("age","std"),
            female=("gender", lambda x: (x.str.lower()=="female").sum()),
            male=("gender", lambda x: (x.str.lower()=="male").sum()))
       .round(1)
)

In [ ]:
meta["rin_num"] = pd.to_numeric(meta["rin"], errors="coerce")
meta["pmi_num"] = pd.to_numeric(meta["post_mortem_interval"], errors="coerce")

print("PMI units:", meta["post_mortem_interval_unit"].unique())
print("Missing RIN:", meta["rin_num"].isna().sum(),
      "| Missing PMI:", meta["pmi_num"].isna().sum())

display(
    meta.groupby(["disease_group_dme","sample_site"])
        .agg(n=("GSM","size"),
             RIN=("rin_num","mean"),
             PMI=("pmi_num","mean"))
        .round(1)
)

In [ ]:
lib = counts.set_index("ensemblID").sum().rename("library_size")

qc = meta.set_index("sampleID").join(lib)
qc["library_M"] = qc["library_size"] / 1e6

display(
    qc.groupby(["disease_group_dme","sample_site"])["library_M"]
      .agg(["mean","min","max"]).round(1)
)

print("Depth vs RIN:", round(qc["library_M"].corr(qc["rin_num"]), 3))
print("Depth vs PMI:", round(qc["library_M"].corr(qc["pmi_num"]), 3))
print("RIN vs PMI:", round(qc["rin_num"].corr(qc["pmi_num"]), 3))

In [ ]:
X = counts.set_index("ensemblID")

qc["detected_genes"] = (X > 0).sum()
qc["zero_pct"] = (X == 0).mean() * 100

display(
    qc[["donor","eye","sample_site","disease_group_dme",
        "library_M","detected_genes","zero_pct"]]
      .sort_values("detected_genes")
      .head(10)
)

print("Detected-gene range:",
      qc["detected_genes"].min(), "to", qc["detected_genes"].max())

In [ ]:
clin = [c for c in meta.columns
        if any(x in c.lower() for x in ["severity","dme","retinopathy"])]

print(clin)

for c in clin:
    print("\n", c)
    display(pd.crosstab(meta["disease_group_dme"], meta[c], dropna=False))

In [ ]:
eye = meta.drop_duplicates(["donor","eye"]).copy()
eye["drss"] = pd.to_numeric(eye["dr_severity_score"], errors="coerce")

display(pd.crosstab(
    [eye["disease_group_dme"], eye["dme"]],
    eye["drss"], margins=True
))

print("Donors with different DRSS between eyes:",
      (eye.groupby("donor")["drss"].nunique() > 1).sum())

In [ ]:
discordant = eye.groupby("donor")["drss"].nunique()
discordant = discordant[discordant > 1].index

display(
    eye[eye["donor"].isin(discordant)]
    [["donor","eye","disease_group_dme","dme","drss"]]
    .sort_values(["donor","eye"])
)

In [ ]:
samples = counts.columns[1:]

m = meta.set_index("sampleID").loc[samples].copy()
m["age"] = pd.to_numeric(m["age"], errors="coerce")
m["rin"] = pd.to_numeric(m["rin"], errors="coerce")
m["pmi"] = pd.to_numeric(m["post_mortem_interval"], errors="coerce")

print("Samples aligned:", len(m))
print("Unique donors:", m["donor"].nunique())
print("Unique eyes:", m.reset_index()[["donor","eye"]].drop_duplicates().shape[0])
print("Any missing key covariates:",
      m[["donor","eye","sample_site","disease_group_dme",
         "age","gender","rin","pmi"]].isna().any().any())

In [ ]:
X = counts.set_index("ensemblID")
cpm_raw = X.div(X.sum(axis=0), axis=1) * 1e6

for n in [4, 8, 12]:
    keep = (cpm_raw >= 1).sum(axis=1) >= n
    print(f"CPM >= 1 in >= {n} samples: {keep.sum()} genes")

In [ ]:
!wget -q -nc https://ftp.ensembl.org/pub/release-86/gtf/homo_sapiens/Homo_sapiens.GRCh38.86.gtf.gz

import gzip, re

gene_map = {}
with gzip.open("Homo_sapiens.GRCh38.86.gtf.gz", "rt") as f:
    for line in f:
        if "\tgene\t" in line:
            gid = re.search(r'gene_id "([^"]+)"', line)
            name = re.search(r'gene_name "([^"]+)"', line)
            if gid and name:
                gene_map[gid.group(1)] = name.group(1)

mapped = counts["ensemblID"].map(gene_map)

print("Ensembl IDs mapped:", mapped.notna().sum(), "/", len(mapped))
print("Unique gene symbols:", mapped.nunique())
print("Unmapped IDs:", mapped.isna().sum())

In [ ]:
ann = pd.DataFrame({
    "ensemblID": counts["ensemblID"],
    "symbol": mapped
})

dup_sym = ann["symbol"].duplicated(keep=False)

print("Rows with duplicated symbols:", dup_sym.sum())
print("Duplicated gene symbols:", ann.loc[dup_sym, "symbol"].nunique())

display(
    ann.loc[dup_sym]
       .groupby("symbol").size()
       .sort_values(ascending=False)
       .head(15)
       .rename("Ensembl_IDs_per_symbol")
)

In [ ]:
locked_known = [
    "Ifit3","Irf9","Samhd1","Stat1","Stat2","Usp18",
    "Bco2","Gpc1","Gpc2","Gpc3","Rimbp2"
]

# human-symbol capitalization
locked_known = [g.upper() for g in locked_known]

tmp = ann.copy()
tmp["symbol_upper"] = tmp["symbol"].str.upper()

display(
    tmp[tmp["symbol_upper"].isin(locked_known)]
    [["ensemblID","symbol"]]
    .sort_values("symbol")
)

print("Known locked genes with >1 Ensembl ID:")
print(
    tmp[tmp["symbol_upper"].isin(locked_known)]
    .groupby("symbol_upper").size()
    .loc[lambda x: x > 1]
)

In [ ]:
for name in sorted(globals()):
    if any(x in name.lower() for x in
           ["early_up", "transient", "ecm", "gag", "ortholog"]):
        obj = globals()[name]
        try:
            print(name, "->", type(obj).__name__, "len =", len(obj))
        except:
            print(name, "->", type(obj).__name__)

In [ ]:
for i, s in enumerate(In):
    if "early_up_transient_all" in s or "early_ECM_GAG_exploratory" in s:
        print(f"\n--- Cell {i} ---\n{s}")

In [ ]:
import os

hits = []
for root, dirs, files in os.walk("/content"):
    dirs[:] = [d for d in dirs if not d.startswith(".")]
    for f in files:
        if any(x in f.lower() for x in
               ["early", "transient", "ecm", "gag", "signature", "gse328966"]):
            hits.append(os.path.join(root, f))

print("\n".join(hits[:100]) if hits else "NO MATCHING FILES FOUND")

In [ ]:
import numpy as np

keep_qc = (cpm_raw >= 1).sum(axis=1) >= 8
logcpm_qc = np.log2(cpm_raw.loc[keep_qc] + 0.5)

print("QC genes retained:", keep_qc.sum())
print("QC matrix:", logcpm_qc.shape)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=5)
pcs = pca.fit_transform(logcpm_qc.T)

pc = pd.DataFrame(
    pcs, index=logcpm_qc.columns,
    columns=["PC1","PC2","PC3","PC4","PC5"]
).join(qc)

print("Variance explained (%):",
      np.round(pca.explained_variance_ratio_ * 100, 1))

display(
    pc.groupby("sample_site")[["PC1","PC2","PC3"]].mean().round(2)
)

display(
    pc.groupby("disease_group_dme")[["PC1","PC2","PC3"]].mean().round(2)
)

In [ ]:
def eta2(y, g):
    z = pd.DataFrame({"y": y, "g": g}).dropna()
    gm = z.y.mean()
    ssb = z.groupby("g").apply(
        lambda x: len(x) * (x.y.mean() - gm)**2,
        include_groups=False
    ).sum()
    return ssb / ((z.y - gm)**2).sum()

rows = []
for p in ["PC1","PC2","PC3","PC4","PC5"]:
    rows.append({
        "PC": p,
        "site": eta2(pc[p], pc["sample_site"]),
        "disease": eta2(pc[p], pc["disease_group_dme"]),
        "sex": eta2(pc[p], pc["gender"]),
        "donor": eta2(pc[p], pc["donor"]),
        "age_R2": pc[p].corr(pc["age"])**2,
        "RIN_R2": pc[p].corr(pc["rin_num"])**2,
        "PMI_R2": pc[p].corr(pc["pmi_num"])**2
    })

display(pd.DataFrame(rows).set_index("PC").round(3))

In [ ]:
for site in ["Macula", "Periphery"]:
    s = pc[pc["sample_site"] == site]

    print(f"\n{site}")
    display(
        s.groupby("disease_group_dme")[["PC2","PC3"]]
         .mean().round(2)
    )

In [ ]:
stage_order = ["Control","Diabetic","NPDR","NPDR/PDR + DME"]

m["stage"] = pd.Categorical(m["disease_group_dme"],
                            categories=stage_order, ordered=True)
m["site"] = pd.Categorical(m["sample_site"],
                           categories=["Periphery","Macula"])
m["sex"] = pd.Categorical(m["gender"])
m["eye_id"] = m["donor"].astype(str) + "_" + m["eye"].astype(str)

for c in ["age","rin","pmi"]:
    m[c+"_c"] = m[c] - m[c].mean()

print("Samples:", len(m))
print("Donors:", m["donor"].nunique(),
      "| Eyes:", m["eye_id"].nunique())
print(pd.crosstab(m["stage"], m["site"]))

In [ ]:
!pip -q install rpy2
%load_ext rpy2.ipython

%R -q -e 'if (!requireNamespace("edgeR", quietly=TRUE)) {install.packages("BiocManager", repos="https://cloud.r-project.org"); BiocManager::install("edgeR", ask=FALSE, update=FALSE)}'

from rpy2.robjects import pandas2ri, r
from rpy2.robjects.conversion import localconverter

with localconverter(r.default_converter + pandas2ri.converter):
    r.assign("cnt", X.astype(int))

r('''
library(edgeR)
dge <- DGEList(counts=cnt)
dge <- calcNormFactors(dge, method="TMM")
''')

normf = np.array(r('dge$samples$norm.factors'))

print("TMM factors:", np.round(normf, 3))
print("Range:", round(normf.min(),3), "to", round(normf.max(),3))

In [ ]:
!pip -q install rpy2

import rpy2.robjects as ro

ro.r('''
if (!requireNamespace("BiocManager", quietly=TRUE))
    install.packages("BiocManager", repos="https://cloud.r-project.org")

if (!requireNamespace("edgeR", quietly=TRUE))
    BiocManager::install("edgeR", ask=FALSE, update=FALSE)

library(edgeR)
''')

print("edgeR version:",
      ro.r('as.character(packageVersion("edgeR"))')[0])

In [ ]:
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

with localconverter(ro.default_converter + pandas2ri.converter):
    ro.globalenv["cnt"] = X.astype(int)

ro.r("""
dge <- DGEList(counts=cnt)
dge <- calcNormFactors(dge, method="TMM")
""")

normf = np.array(ro.r("dge$samples$norm.factors"))

print("TMM factor range:", round(normf.min(),3), "to", round(normf.max(),3))
print("Median:", round(np.median(normf),3))
print("Most extreme:", np.round(np.sort(normf)[[0,1,-2,-1]],3))

In [ ]:
with localconverter(ro.default_converter + pandas2ri.converter):
    ro.globalenv["meta_r"] = m.reset_index()

ro.r("""
meta_r$stage <- factor(meta_r$stage,
  levels=c("Control","Diabetic","NPDR","NPDR/PDR + DME"))
meta_r$site <- factor(meta_r$site, levels=c("Periphery","Macula"))
meta_r$sex <- factor(meta_r$sex)

design <- model.matrix(
  ~ stage*site + sex + age_c + rin_c + pmi_c,
  data=meta_r
)

cat("Design:", nrow(design), "samples x", ncol(design), "terms\n")
cat("Rank:", qr(design)$rank, "/", ncol(design), "\n")
cat("Condition number:", round(kappa(design),1), "\n")
""")

In [ ]:
for c in ["age","rin","pmi"]:
    m[c+"_z"] = (m[c] - m[c].mean()) / m[c].std()

model_meta = m.copy()
model_meta["sampleID"] = model_meta.index

model_meta = model_meta[
    ["sampleID","donor","stage","site","sex","age_z","rin_z","pmi_z"]
].reset_index(drop=True)

with localconverter(ro.default_converter + pandas2ri.converter):
    ro.globalenv["model_meta"] = model_meta

ro.r("""
design <- model.matrix(
  ~ stage*site + sex + age_z + rin_z + pmi_z,
  data=model_meta
)
cat("Rank:", qr(design)$rank, "/", ncol(design), "\n")
cat("Condition number:", round(kappa(design),1), "\n")
""")

In [ ]:
from rpy2.robjects.vectors import StrVector

ro.globalenv["qc_genes"] = StrVector(logcpm_qc.index.astype(str).tolist())

ro.r(r'''
library(limma)

y <- dge[rownames(dge) %in% qc_genes, , keep.lib.sizes=FALSE]
y <- normLibSizes(y)

v0 <- voom(y, design, plot=FALSE)
dc <- duplicateCorrelation(v0, design, block=model_meta$donor)

v <- voom(y, design, plot=FALSE,
          block=model_meta$donor,
          correlation=dc$consensus.correlation)

fit <- lmFit(v, design,
             block=model_meta$donor,
             correlation=dc$consensus.correlation)
fit <- eBayes(fit)

cat("Genes modeled:", nrow(v), "\n")
cat("Donor correlation:", round(dc$consensus.correlation, 3), "\n")
cat("Residual df:", paste(range(fit$df.residual), collapse=" to "), "\n")
''')

In [ ]:
print(list(ro.r("colnames(design)")))

In [ ]:
ro.r(r'''
model_meta$stage2 <- factor(as.character(model_meta$stage),
  levels=c("Control","Diabetic","NPDR","NPDR/PDR + DME"),
  labels=c("Control","Diabetic","NPDR","Advanced"))

model_meta$site2 <- factor(as.character(model_meta$site),
  levels=c("Periphery","Macula"))

model_meta$group <- interaction(model_meta$stage2, model_meta$site2, sep="_")

design <- model.matrix(
  ~ 0 + group + sex + age_z + rin_z + pmi_z,
  data=model_meta
)
colnames(design) <- make.names(colnames(design))

v0 <- voom(y, design, plot=FALSE)
dc <- duplicateCorrelation(v0, design, block=model_meta$donor)

v <- voom(y, design, plot=FALSE,
          block=model_meta$donor,
          correlation=dc$consensus.correlation)

fit <- eBayes(lmFit(v, design,
                    block=model_meta$donor,
                    correlation=dc$consensus.correlation))

cat("Rank:", qr(design)$rank, "/", ncol(design), "\n")
cat("Donor correlation:", round(dc$consensus.correlation,3), "\n")
print(colnames(design))
''')

In [ ]:
ro.r(r'''
contr <- makeContrasts(
  DMvsCTRL_P  = groupDiabetic_Periphery - groupControl_Periphery,
  NPDRvsCTRL_P = groupNPDR_Periphery - groupControl_Periphery,
  ADVvsCTRL_P  = groupAdvanced_Periphery - groupControl_Periphery,
  NPDRvsDM_P   = groupNPDR_Periphery - groupDiabetic_Periphery,
  ADVvsNPDR_P  = groupAdvanced_Periphery - groupNPDR_Periphery,

  DMvsCTRL_M  = groupDiabetic_Macula - groupControl_Macula,
  NPDRvsCTRL_M = groupNPDR_Macula - groupControl_Macula,
  ADVvsCTRL_M  = groupAdvanced_Macula - groupControl_Macula,
  NPDRvsDM_M   = groupNPDR_Macula - groupDiabetic_Macula,
  ADVvsNPDR_M  = groupAdvanced_Macula - groupNPDR_Macula,
  levels=design
)

fit0 <- lmFit(v, design,
              block=model_meta$donor,
              correlation=dc$consensus.correlation)

fitC <- eBayes(contrasts.fit(fit0, contr))

print(colnames(contr))
''')

In [ ]:
de_counts = ro.r(r'''
out <- t(sapply(colnames(contr), function(k) {
  tt <- topTable(fitC, coef=k, number=Inf, sort.by="none")
  c(
    UP   = sum(tt$adj.P.Val < 0.05 & tt$logFC > 0),
    DOWN = sum(tt$adj.P.Val < 0.05 & tt$logFC < 0),
    TOTAL= sum(tt$adj.P.Val < 0.05)
  )
}))
out
''')

display(pd.DataFrame(
    np.array(de_counts),
    index=list(ro.r("colnames(contr)")),
    columns=["UP","DOWN","TOTAL"]
).astype(int))

In [ ]:
sigs = {
    "IFN6": ["IFIT3","IRF9","SAMHD1","STAT1","STAT2","USP18"],
    "Retinoid5": ["BCO2","GPC1","GPC2","GPC3","RIMBP2"]
}

rows = []
for s, genes in sigs.items():
    for g in genes:
        ids = ann.loc[ann["symbol"] == g, "ensemblID"].tolist()
        rows.append([s, g, ids[0] if ids else None,
                     bool(ids and ids[0] in logcpm_qc.index)])

sig_audit = pd.DataFrame(
    rows, columns=["signature","gene","ensemblID","modeled"]
)

display(sig_audit)
print(sig_audit.groupby("signature")["modeled"].agg(["sum","count"]))

In [ ]:
gid = "ENSG00000184979"

print("Nonzero samples:", (X.loc[gid] > 0).sum(), "/ 79")
print("CPM >= 1 samples:", (cpm_raw.loc[gid] >= 1).sum(), "/ 79")
print("Median CPM:", round(cpm_raw.loc[gid].median(), 3))
print("Max CPM:", round(cpm_raw.loc[gid].max(), 3))
print("Total counts:", int(X.loc[gid].sum()))

In [ ]:
qc_plus = list(logcpm_qc.index) + ["ENSG00000184979"]
ro.globalenv["qc_plus"] = StrVector(qc_plus)

ro.r(r'''
y_sig <- dge[rownames(dge) %in% qc_plus, , keep.lib.sizes=FALSE]
y_sig <- normLibSizes(y_sig)

v_sig <- voom(y_sig, design, plot=FALSE,
              block=model_meta$donor,
              correlation=dc$consensus.correlation)

fit_sig <- lmFit(v_sig, design,
                 block=model_meta$donor,
                 correlation=dc$consensus.correlation)

fitC_sig <- eBayes(contrasts.fit(fit_sig, contr))

cat("Genes modeled:", nrow(v_sig), "\n")
cat("USP18 retained:", "ENSG00000184979" %in% rownames(v_sig), "\n")
cat("Donor correlation used:", round(dc$consensus.correlation,3), "\n")
''')

In [ ]:
sig_ids = {
    "IFN6": ["ENSG00000119917","ENSG00000213928","ENSG00000101347",
             "ENSG00000115415","ENSG00000170581","ENSG00000184979"],
    "Retinoid5": ["ENSG00000197580","ENSG00000063660","ENSG00000213420",
                  "ENSG00000147257","ENSG00000060709"]
}

rows = []
for sig, ids in sig_ids.items():
    ro.globalenv["ids_now"] = StrVector(ids)
    z = np.array(ro.r("fitC_sig$coefficients[ids_now,,drop=FALSE]"))
    for j, con in enumerate(list(ro.r("colnames(contr)"))):
        vals = z[:, j]
        rows.append([sig, con, np.median(vals), np.mean(vals),
                     sum(vals > 0), len(vals)])

prof = pd.DataFrame(rows, columns=
    ["signature","contrast","median_logFC","mean_logFC","n_up","n_genes"])

display(prof.round(3))

In [ ]:
sig_stats = ro.r(r'''
E <- v_sig$E
Z <- t(scale(t(E)))

S <- rbind(
  IFN6 = colMeans(Z[c("ENSG00000119917","ENSG00000213928",
                      "ENSG00000101347","ENSG00000115415",
                      "ENSG00000170581","ENSG00000184979"),]),
  Retinoid5 = colMeans(Z[c("ENSG00000197580","ENSG00000063660",
                           "ENSG00000213420","ENSG00000147257",
                           "ENSG00000060709"),])
)

fs <- lmFit(S, design, block=model_meta$donor,
            correlation=dc$consensus.correlation)
fc <- contrasts.fit(fs, contr)

SE <- fc$stdev.unscaled * fc$sigma
T  <- fc$coefficients / SE
DF <- matrix(fc$df.residual, nrow=nrow(T), ncol=ncol(T))
P  <- 2*pt(-abs(T), df=DF)

res <- data.frame(
  signature=rep(rownames(T), times=ncol(T)),
  contrast=rep(colnames(T), each=nrow(T)),
  effect_SD=as.vector(fc$coefficients),
  p=as.vector(P)
)
res$FDR <- p.adjust(res$p, method="BH")
res
''')

display(pd.DataFrame(sig_stats).round(4))

In [ ]:
keep76 = model_meta["sampleID"].isin(cpm_samples).values

ro.globalenv["keep76"] = ro.BoolVector(keep76)

ro.r(r'''
m76 <- model_meta[keep76,]
y76 <- dge[, keep76]

design76 <- model.matrix(
  ~ 0 + interaction(stage2,site2,sep="_") + sex + age_z + rin_z + pmi_z,
  data=m76
)
colnames(design76) <- make.names(colnames(design76))

y76 <- normLibSizes(y76)
v0_76 <- voom(y76, design76, plot=FALSE)
dc76 <- duplicateCorrelation(v0_76, design76, block=m76$donor)

cat("Samples:", ncol(y76), "\n")
cat("Donor correlation:", round(dc76$consensus.correlation,3), "\n")
cat("Rank:", qr(design76)$rank, "/", ncol(design76), "\n")
''')

In [ ]:
ecm13 = [
    "ADAMTS5","CAPN5","COL15A1","COL27A1","COL9A2",
    "GPC1","GPC2","GPC3","ITGAV","ITGB8","NAGLU","P3H2","TLL1"
]

modeled_ids = set(map(str, list(ro.r("rownames(v_sig)"))))

rows = []
for g in ecm13:
    ids = ann.loc[ann["symbol"] == g, "ensemblID"].tolist()
    rows.append([
        g,
        ids[0] if ids else None,
        bool(ids and ids[0] in modeled_ids)
    ])

ecm_audit = pd.DataFrame(
    rows, columns=["gene","ensemblID","modeled"]
)

display(ecm_audit)
print("ECM/GAG modeled:", ecm_audit["modeled"].sum(), "/ 13")

In [ ]:
gid = "ENSG00000204291"

print("Nonzero samples:", (X.loc[gid] > 0).sum(), "/ 79")
print("CPM >= 1 samples:", (cpm_raw.loc[gid] >= 1).sum(), "/ 79")
print("Median CPM:", round(cpm_raw.loc[gid].median(), 3))
print("Max CPM:", round(cpm_raw.loc[gid].max(), 3))
print("Total counts:", int(X.loc[gid].sum()))

In [ ]:
force = ["ENSG00000184979", "ENSG00000204291"]  # USP18, COL15A1
qc_sig = list(dict.fromkeys(list(logcpm_qc.index) + force))
ro.globalenv["qc_sig"] = StrVector(qc_sig)

ro.r(r'''
y3 <- dge[rownames(dge) %in% qc_sig,,keep.lib.sizes=FALSE]
y3 <- normLibSizes(y3)

v3 <- voom(y3, design, plot=FALSE,
           block=model_meta$donor,
           correlation=dc$consensus.correlation)

Z <- t(scale(t(v3$E)))

S <- rbind(
 IFN6=colMeans(Z[c("ENSG00000119917","ENSG00000213928","ENSG00000101347",
                    "ENSG00000115415","ENSG00000170581","ENSG00000184979"),]),
 Retinoid5=colMeans(Z[c("ENSG00000197580","ENSG00000063660","ENSG00000213420",
                        "ENSG00000147257","ENSG00000060709"),]),
 ECM13=colMeans(Z[c("ENSG00000154736","ENSG00000149260","ENSG00000204291",
                    "ENSG00000196739","ENSG00000049089","ENSG00000063660",
                    "ENSG00000213420","ENSG00000147257","ENSG00000138448",
                    "ENSG00000105855","ENSG00000108784","ENSG00000090530",
                    "ENSG00000038295"),])
)

fs <- lmFit(S, design, block=model_meta$donor,
            correlation=dc$consensus.correlation)
fc <- contrasts.fit(fs, contr)

SE <- fc$stdev.unscaled * fc$sigma
T <- fc$coefficients / SE
DF <- matrix(fc$df.residual, nrow=nrow(T), ncol=ncol(T))
P <- 2*pt(-abs(T), DF)

res3 <- data.frame(
 signature=rep(rownames(T), times=ncol(T)),
 contrast=rep(colnames(T), each=nrow(T)),
 effect_SD=as.vector(fc$coefficients),
 p=as.vector(P)
)
res3$FDR <- p.adjust(res3$p, "BH")
''')

with localconverter(ro.default_converter + pandas2ri.converter):
    res3 = ro.conversion.rpy2py(ro.r("res3"))

display(res3.round(4))

In [ ]:
keep76 = model_meta["sampleID"].isin(cpm_samples).tolist()
ro.globalenv["keep76"] = ro.BoolVector(keep76)

ro.r(r'''
m76 <- model_meta[keep76,]
y76 <- dge[rownames(dge) %in% qc_sig, keep76, keep.lib.sizes=FALSE]
y76 <- normLibSizes(y76)

m76$group <- interaction(m76$stage2, m76$site2, sep="_")
d76 <- model.matrix(~0 + group + sex + age_z + rin_z + pmi_z, data=m76)
colnames(d76) <- make.names(colnames(d76))

v0 <- voom(y76, d76, plot=FALSE)
dc76 <- duplicateCorrelation(v0, d76, block=m76$donor)
v76 <- voom(y76, d76, plot=FALSE,
            block=m76$donor, correlation=dc76$consensus.correlation)

Z <- t(scale(t(v76$E)))

S <- rbind(
 IFN6=colMeans(Z[c("ENSG00000119917","ENSG00000213928","ENSG00000101347",
                    "ENSG00000115415","ENSG00000170581","ENSG00000184979"),]),
 Retinoid5=colMeans(Z[c("ENSG00000197580","ENSG00000063660","ENSG00000213420",
                        "ENSG00000147257","ENSG00000060709"),]),
 ECM13=colMeans(Z[c("ENSG00000154736","ENSG00000149260","ENSG00000204291",
                    "ENSG00000196739","ENSG00000049089","ENSG00000063660",
                    "ENSG00000213420","ENSG00000147257","ENSG00000138448",
                    "ENSG00000105855","ENSG00000108784","ENSG00000090530",
                    "ENSG00000038295"),])
)

C76 <- makeContrasts(
 DMvsCTRL_P=groupDiabetic_Periphery-groupControl_Periphery,
 NPDRvsCTRL_P=groupNPDR_Periphery-groupControl_Periphery,
 ADVvsCTRL_P=groupAdvanced_Periphery-groupControl_Periphery,
 NPDRvsDM_P=groupNPDR_Periphery-groupDiabetic_Periphery,
 ADVvsNPDR_P=groupAdvanced_Periphery-groupNPDR_Periphery,
 DMvsCTRL_M=groupDiabetic_Macula-groupControl_Macula,
 NPDRvsCTRL_M=groupNPDR_Macula-groupControl_Macula,
 ADVvsCTRL_M=groupAdvanced_Macula-groupControl_Macula,
 NPDRvsDM_M=groupNPDR_Macula-groupDiabetic_Macula,
 ADVvsNPDR_M=groupAdvanced_Macula-groupNPDR_Macula,
 levels=d76
)

f <- lmFit(S, d76, block=m76$donor, correlation=dc76$consensus.correlation)
fc <- contrasts.fit(f, C76)
SE <- fc$stdev.unscaled * fc$sigma
P <- 2*pt(-abs(fc$coefficients/SE), fc$df.residual)

sens76 <- data.frame(
 signature=rep(rownames(fc$coefficients), times=10),
 contrast=rep(colnames(C76), each=3),
 effect_SD=as.vector(fc$coefficients),
 p=as.vector(P)
)
sens76$FDR <- p.adjust(sens76$p, "BH")

cat("Samples:", ncol(y76),
    "| donor corr:", round(dc76$consensus.correlation,3), "\n")
print(sens76)
''')

In [ ]:
import json, os, pandas as pd
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/DR_Muller_AdaptiveReactive")

if not ROOT.exists():
    from google.colab import drive
    drive.mount("/content/drive")

with open(ROOT/"signatures/GSE328966_LOCKED_discovery_signatures.json") as f:
    locked = json.load(f)

orth = pd.read_csv(
    ROOT/"02_Human_scRNA_2026/Mouse_to_Human_orthologs_LOCKED.csv"
)

early184 = locked["early_up_transient_all"]

present_mouse = orth[
    orth["mouse_symbol"].isin(early184) &
    orth["human_symbol"].isin(ann["symbol"])
]["mouse_symbol"].nunique()

present_human = orth[
    orth["mouse_symbol"].isin(early184) &
    orth["human_symbol"].isin(ann["symbol"])
]["human_symbol"].nunique()

print("Frozen mouse genes:", len(early184))
print("Mouse genes with human ortholog present:", present_mouse)
print("Unique human ortholog genes:", present_human)

In [ ]:
early_map = (
    orth[orth["mouse_symbol"].isin(early184)]
    .dropna(subset=["human_symbol"])
    .drop_duplicates("human_symbol")
    .merge(ann, left_on="human_symbol", right_on="symbol")
)

modeled = set(map(str, ro.r("rownames(v3)")))
early_map["modeled"] = early_map["ensemblID"].isin(modeled)

print("Human orthologs present:", len(early_map))
print("Already modeled:", early_map["modeled"].sum())
print("Filtered out:", (~early_map["modeled"]).sum())

display(
    early_map.loc[~early_map["modeled"],
                  ["mouse_symbol","human_symbol","ensemblID"]]
)

In [ ]:
dup = (
    early_map.groupby("human_symbol")
    .filter(lambda x: len(x) > 1)
    [["mouse_symbol","human_symbol","ensemblID","modeled"]]
    .sort_values(["human_symbol","ensemblID"])
)

print("Rows:", len(early_map))
print("Unique human symbols:", early_map["human_symbol"].nunique())
print("Symbols with >1 Ensembl ID:", dup["human_symbol"].nunique())

display(dup)

In [ ]:
tmp = early_map.copy()
tmp["total_counts"] = tmp["ensemblID"].map(X.sum(axis=1))

early168 = (
    tmp.sort_values("total_counts", ascending=False)
       .drop_duplicates("human_symbol")
       .copy()
)

modeled = set(map(str, ro.r("rownames(v3)")))
early168["modeled"] = early168["ensemblID"].isin(modeled)

print("Unique Early184 human genes:", len(early168))
print("Already modeled:", early168["modeled"].sum())
print("Filtered out:", (~early168["modeled"]).sum())

print("\nDuplicate choices:")
display(
    early168[early168["human_symbol"].isin(["CLN3","CYB561D2"])]
    [["mouse_symbol","human_symbol","ensemblID","total_counts","modeled"]]
)

low = early168.loc[~early168["modeled"],
                   ["mouse_symbol","human_symbol","ensemblID"]].copy()

low["nonzero"] = low["ensemblID"].map(lambda g: int((X.loc[g] > 0).sum()))
low["CPM_ge_1"] = low["ensemblID"].map(lambda g: int((cpm_raw.loc[g] >= 1).sum()))
low["median_CPM"] = low["ensemblID"].map(lambda g: cpm_raw.loc[g].median())
low["total_counts"] = low["ensemblID"].map(lambda g: int(X.loc[g].sum()))

display(low)

In [ ]:
# Final measurable Early184 human representation:
# retain 163 modeled + OLFM4/TEN1/IRGM; exclude only unmeasurable CCL21/PIGY

extra_early = ["ENSG00000102837", "ENSG00000257949", "ENSG00000237693"]
early166 = early168[
    (~early168["human_symbol"].isin(["CCL21","PIGY"]))
]["ensemblID"].tolist()

force_all = list(dict.fromkeys(
    ["ENSG00000184979", "ENSG00000204291"] + extra_early
))
genes4 = list(dict.fromkeys(list(logcpm_qc.index) + force_all))

ro.globalenv["genes4"] = StrVector(genes4)
ro.globalenv["early166"] = StrVector(early166)

final40 = ro.r(r'''
y4 <- dge[rownames(dge) %in% genes4,,keep.lib.sizes=FALSE]
y4 <- normLibSizes(y4)

v4 <- voom(y4, design, plot=FALSE,
           block=model_meta$donor,
           correlation=dc$consensus.correlation)

Z <- t(scale(t(v4$E)))

S4 <- rbind(
 Early166 = colMeans(Z[early166,]),
 IFN6 = colMeans(Z[c("ENSG00000119917","ENSG00000213928",
                      "ENSG00000101347","ENSG00000115415",
                      "ENSG00000170581","ENSG00000184979"),]),
 Retinoid5 = colMeans(Z[c("ENSG00000197580","ENSG00000063660",
                           "ENSG00000213420","ENSG00000147257",
                           "ENSG00000060709"),]),
 ECM13 = colMeans(Z[c("ENSG00000154736","ENSG00000149260",
                       "ENSG00000204291","ENSG00000196739",
                       "ENSG00000049089","ENSG00000063660",
                       "ENSG00000213420","ENSG00000147257",
                       "ENSG00000138448","ENSG00000105855",
                       "ENSG00000108784","ENSG00000090530",
                       "ENSG00000038295"),])
)

f <- lmFit(S4, design, block=model_meta$donor,
           correlation=dc$consensus.correlation)
fc <- contrasts.fit(f, contr)

SE <- fc$stdev.unscaled * fc$sigma
T <- fc$coefficients / SE
DF <- matrix(fc$df.residual, nrow=nrow(T), ncol=ncol(T))
P <- 2*pt(-abs(T), DF)

res4 <- data.frame(
 signature=rep(rownames(T), times=ncol(T)),
 contrast=rep(colnames(T), each=nrow(T)),
 effect_meanZ=as.vector(fc$coefficients),
 p=as.vector(P)
)

res4$FDR <- p.adjust(res4$p, "BH")

cat("Early genes tested:", length(early166), "\n")
res4
''')

display(pd.DataFrame(final40).round(4))

In [ ]:
with localconverter(ro.default_converter + pandas2ri.converter):
    res4_df = ro.conversion.rpy2py(ro.r("res4"))

res4_df = res4_df.reset_index(drop=True)

display(
    res4_df[
        ["signature","contrast","effect_meanZ","p","FDR"]
    ].round(4)
)

In [ ]:
# Pull the 76-sample sensitivity table from R into Python
with localconverter(ro.default_converter + pandas2ri.converter):
    sens76_df = ro.conversion.rpy2py(ro.r("sens76"))

OUT = ROOT / "03_GSE160306_human_validation"
OUT.mkdir(parents=True, exist_ok=True)

res4_df.to_csv(
    OUT / "GSE160306_FINAL_locked_signature_tests_79samples.csv",
    index=False
)

sens76_df.to_csv(
    OUT / "GSE160306_FINAL_locked_signature_sensitivity_76samples.csv",
    index=False
)

early168.to_csv(
    OUT / "GSE160306_Early184_human_ortholog_coverage.csv",
    index=False
)

print("GSE160306 CLOSED")